# TRELLIS.2 notebook quick start

1. Select the Jupyter kernel for the `phase2_3d` conda environment and confirm the project root is `~/OEC`.
2. Restart the kernel if a setup cell says an already-imported package was upgraded. Run the notebook from the first cell in order; do not jump into the studio with a fresh kernel.
3. The setup cell uses the active Python interpreter and must pass OEC's fresh-quota/storage admission gate before installing dependencies, compiling extensions, or downloading model weights. If admission fails, stop there and refresh quota telemetry; do not bypass the guard.
4. The Gradio UI launches on port 7862 with public sharing disabled by default.

### SSH tunnels

Print the compute node name from the notebook with `!hostname`. Run the command on your local computer and replace `<COMPUTE_NODE>` with that exact host name:

```bash
ssh -N -L 5402:<COMPUTE_NODE>:8888 rahkus@login.npsf.cdac.in
ssh -N -L 7862:<COMPUTE_NODE>:7862 rahkus@login.npsf.cdac.in
```

Keep these tunnels in separate terminals while using Jupyter and the Gradio interface.


In [ ]:
# Set GPU visibility before importing torch or any CUDA library.
import os
TARGET_GPU = os.environ.get("AIEROS_TRELLIS_GPU", "0")
os.environ["CUDA_VISIBLE_DEVICES"] = TARGET_GPU
os.environ["no_proxy"] = "localhost,127.0.0.1,::1,0.0.0.0"
os.environ["NO_PROXY"] = os.environ["no_proxy"]
import core_config
from core_config import SYSTEM_DIRS
print(f"[SYSTEM] TRELLIS GPU visibility configured before CUDA imports: {TARGET_GPU}")
print("[SYSTEM] Shared CUDA/runtime package removal is disabled.")


In [2]:
# Read-only PyTorch/CUDA compatibility gate; no forced CUDA-library reinstall.
import torch
print(f"PyTorch: {torch.__version__} | Compiled CUDA: {torch.version.cuda} | cuDNN: {torch.backends.cudnn.version()}")
assert torch.__version__.startswith("2.6.0"), f"Expected PyTorch 2.6.0, got {torch.__version__}"
assert torch.version.cuda == "12.4", f"Expected CUDA 12.4 build, got {torch.version.cuda}"


ERROR: Could not find a version that satisfies the requirement nvidia-cudnn-cu12==9.1.0.70 (from versions: none)
ERROR: No matching distribution found for nvidia-cudnn-cu12==9.1.0.70


In [3]:
!python -c "import torch; print(f'PyTorch: {torch.__version__} | Compiled CUDA: {torch.version.cuda} | Active cuDNN: {torch.backends.cudnn.version()}')"

PyTorch: 2.6.0+cu124 | Compiled CUDA: 12.4 | Active cuDNN: 90100


In [4]:
!pip list | grep -iE 'cu13|cuda-toolkit'

In [5]:
!pip list | grep -i cu12

nvidia-cublas-cu12        12.4.5.8
nvidia-cuda-cupti-cu12    12.4.127
nvidia-cuda-nvrtc-cu12    12.4.127
nvidia-cuda-runtime-cu12  12.4.127
nvidia-cudnn-cu12         9.1.0.70
nvidia-cufft-cu12         11.2.1.3
nvidia-curand-cu12        10.3.5.147
nvidia-cusolver-cu12      11.6.1.9
nvidia-cusparse-cu12      12.3.1.170
nvidia-cusparselt-cu12    0.6.2
nvidia-nccl-cu12          2.21.5
nvidia-nvjitlink-cu12     12.4.127
nvidia-nvtx-cu12          12.4.127
torch                     2.6.0+cu124
torchvision               0.21.0+cu124


In [ ]:
import os

# ==============================================================================
# GLOBAL CONFIGURATION & TOGGLES
# ==============================================================================
FORCE_RECOMPILE = False    # Set to True only if you need to wipe and rebuild C++ extensions
ENABLE_DISK_PURGE = False  # Set False for safety. Leaves models on SSD for mmap.
TARGET_GPU = os.environ.get("AIEROS_TRELLIS_GPU", "0")

# 1. ISOLATE GPU: MUST BE THE ABSOLUTE FIRST LINE EXECUTED
if os.environ.get("CUDA_VISIBLE_DEVICES") != TARGET_GPU:
    raise RuntimeError("Set AIEROS_TRELLIS_GPU before the first notebook cell; restart the kernel after changing GPU.")
print(f"[SYSTEM] Notebook is using preselected visible GPU: {TARGET_GPU}")

# 2. FIX GRADIO PROXY CRASH: Force Python to bypass proxy for localhost pings
os.environ["no_proxy"] = "localhost, 127.0.0.1, ::1, 0.0.0.0"
os.environ["NO_PROXY"] = "localhost, 127.0.0.1, ::1, 0.0.0.0"

# 3. THE FROZEN RULE: Core Config
import core_config
print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")
from core_config import SYSTEM_DIRS

# 4. DISABLE BENCHMARKING (Prevents cuDNN workspace fragmentation panics)
#import torch
#torch.backends.cudnn.benchmark = False

In [7]:
# Read-only process inspection; never kill a hard-coded PID.
!ps -u "$USER" -o pid,etime,cmd | grep -E '[g]radio|[p]ipeline_trellis' || true


/bin/bash: line 1: kill: (9) - No such process
/bin/bash: line 1: kill: (2386856) - No such process


In [8]:
!nvidia-smi

Sun Sep 13 00:20:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   31C    P0             50W /  400W |       1MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [9]:
# Verify notebook UI packages without upgrading them during model setup.
import importlib.metadata
for package in ("jupyter", "ipywidgets", "ipykernel"):
    try:
        print(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}=MISSING (install only in phase2_3d after explicit approval)")


In [ ]:
import os
import sys
import subprocess
import shutil
import importlib.util
from core_config import SYSTEM_DIRS
from storage_manager import storage_preflight_path

# Dependency builds and package downloads count against the shared home quota.
setup_budget_gb = max(1.0, float(os.environ.get("AIEROS_TRELLIS_SETUP_BUDGET_GB", "8.0")))
ok, reason = storage_preflight_path(
    SYSTEM_DIRS["cache"],
    required_gb=setup_budget_gb,
    require_fresh_quota=True,
    minimum_free_gb=0.5,
)
if not ok:
    raise RuntimeError(f"TRELLIS setup denied by storage policy: {reason}")

def run_cmd(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)

def is_installed(pkg_name):
    return importlib.util.find_spec(pkg_name) is not None

# ==============================================================================
# 0. THE NUCLEAR UNINSTALL & CUDA 13 PURGE
# ==============================================================================
print("0. Preserving existing CUDA runtime packages; no automatic CUDA 13 package removal.")

if FORCE_RECOMPILE:
    raise RuntimeError("Automatic package uninstalls are disabled in the shared phase2_3d environment.")

# ==============================================================================
# 1. IRONCLAD PYTORCH 2.6.0 & COMPATIBLE XFORMERS INSTALLATION
# ==============================================================================
print("1. Verifying PyTorch 2.6.0 (CUDA 12.4)...")
_torch_probe = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__); print(torch.version.cuda)"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,
)
_torch_ok = _torch_probe.returncode == 0 and "2.6.0" in _torch_probe.stdout and "12.4" in _torch_probe.stdout
if not _torch_ok:
    run_cmd([sys.executable, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
else:
    print("Existing PyTorch CUDA 12.4 runtime matches the required pin; skipping reinstall.")

# Keep PyTorch's CUDA 12.4 transitive libraries pinned by the PyTorch wheel.
# Do not force-reinstall cuDNN/cuBLAS/NVRTC/NCCL independently.
# xformers 0.0.29.post3 is the release line built for PyTorch 2.6.0.
_xformers_version = None
try:
    _xformers_version = importlib.metadata.version("xformers")
except importlib.metadata.PackageNotFoundError:
    pass
if _xformers_version != "0.0.29.post2":
    print("1b. Installing CUDA-enabled xFormers matched to PyTorch 2.6.0+cu124...")
    run_cmd([
        sys.executable, "-m", "pip", "install", "--force-reinstall",
        "--no-deps", "--no-cache-dir", "xformers==0.0.29.post2",
        "--index-url", "https://download.pytorch.org/whl/cu124",
    ])
else:
    print("Existing xFormers pin is correct; validating its CUDA kernels without reinstalling.")

print("Verifying installed xFormers CUDA kernels...")
_xf = subprocess.run(
    [sys.executable, "-m", "xformers.info"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,
)
print(_xf.stdout)
if _xf.returncode != 0:
    raise RuntimeError("xformers.info failed after the pinned wheel installation.")
_xf_fields = {}
for _line in _xf.stdout.splitlines():
    if ":" in _line:
        _key, _value = _line.split(":", 1)
        _xf_fields[_key.strip()] = _value.strip()
if (
    _xf_fields.get("build.cuda_version") != "1204"
    or _xf_fields.get("pytorch.cuda") != "available"
    or _xf_fields.get("memory_efficient_attention.cutlassF-pt") != "available"
):
    raise RuntimeError(
        "xFormers CUDA validation failed. Expected CUDA 12.4 attention kernels "
        "for PyTorch 2.6.0; do not continue with TRELLIS inference."
    )
print("XFORMERS_CUDA_VALIDATION=PASS")

# ==============================================================================
# 2. OFFICIAL SETUP.SH DEPENDENCIES
# ==============================================================================
print("2. Installing Basic Dependencies...")
run_cmd([
    sys.executable, "-m", "pip", "install", 
    "imageio", "imageio-ffmpeg", "tqdm", "easydict", "opencv-python-headless", 
    "ninja", "trimesh", "transformers==4.56.1", "gradio==6.0.1", "tensorboard", "pandas", 
    "lpips", "zstandard", "kornia", "timm", "pillow"
])

run_cmd([sys.executable, "-m", "pip", "install", "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8"])

# ==============================================================================
# 3. C++ EXTENSIONS COMPILATION
# ==============================================================================
extensions_exist = is_installed("nvdiffrast") and is_installed("nvdiffrec_render") and is_installed("cumesh") and is_installed("flex_gemm") and is_installed("o_voxel")

if not FORCE_RECOMPILE and extensions_exist:
    print("3. ✅ 3D C++ Extensions already detected! Skipping heavy compilation.")
else:
    print("3. Compiling C++ Extensions...")
    EXT_DIR = os.path.join(SYSTEM_DIRS['cache'], "trellis2_extensions")
    os.makedirs(EXT_DIR, exist_ok=True)

    nd_dir = os.path.join(EXT_DIR, "nvdiffrast")
    if not os.path.isdir(nd_dir):
        run_cmd(["git", "clone", "-b", "v0.4.0", "https://github.com/NVlabs/nvdiffrast.git", nd_dir])
    else:
        print(f"Reusing existing extension source checkout: {nd_dir}")
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", nd_dir, "--no-build-isolation"])

    nr_dir = os.path.join(EXT_DIR, "nvdiffrec")
    if not os.path.isdir(nr_dir):
        run_cmd(["git", "clone", "-b", "renderutils", "https://github.com/JeffreyXiang/nvdiffrec.git", nr_dir])
    else:
        print(f"Reusing existing extension source checkout: {nr_dir}")
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", nr_dir, "--no-build-isolation"])

    cm_dir = os.path.join(EXT_DIR, "CuMesh")
    if not os.path.isdir(cm_dir):
        run_cmd(["git", "clone", "https://github.com/JeffreyXiang/CuMesh.git", cm_dir, "--recursive"])
    else:
        print(f"Reusing existing extension source checkout: {cm_dir}")
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", cm_dir, "--no-build-isolation"])

    fg_dir = os.path.join(EXT_DIR, "FlexGEMM")
    if not os.path.isdir(fg_dir):
        run_cmd(["git", "clone", "https://github.com/JeffreyXiang/FlexGEMM.git", fg_dir, "--recursive"])
    else:
        print(f"Reusing existing extension source checkout: {fg_dir}")
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", fg_dir, "--no-build-isolation"])

t2_dir = os.path.join(SYSTEM_DIRS['models'], "TRELLIS.2")
if FORCE_RECOMPILE or not os.path.exists(os.path.join(t2_dir, "setup.sh")):
    if os.path.exists(t2_dir):
        raise RuntimeError("Existing TRELLIS.2 source checkout is incomplete; refusing to delete or replace it automatically.")
    run_cmd(["git", "clone", "-b", "main", "https://github.com/microsoft/TRELLIS.2.git", t2_dir, "--recursive"])

if FORCE_RECOMPILE or not is_installed("o_voxel"):
    run_cmd([sys.executable, "-m", "pip", "install", "--no-cache-dir", os.path.join(t2_dir, "o-voxel"), "--no-build-isolation"])

sys.path.insert(0, t2_dir)
print("Environment Ready! TRELLIS.2 mapped and ready.")

In [ ]:
import core_config
from core_config import SYSTEM_DIRS

import os
TRELLIS_REPO_ID = "microsoft/TRELLIS.2-4B"
TRELLIS_HF_CACHE_ROOT = os.path.join(
    SYSTEM_DIRS["models"], "hub", "models--microsoft--TRELLIS.2-4B"
)
TRELLIS_SNAPSHOTS_DIR = os.path.join(TRELLIS_HF_CACHE_ROOT, "snapshots")
TRELLIS_LOCAL_CACHE_AVAILABLE = (
    os.path.isdir(TRELLIS_SNAPSHOTS_DIR) and bool(os.listdir(TRELLIS_SNAPSHOTS_DIR))
)
# Cache-first snapshot resolution is used below; keep the UI kernel able to access other models.

import sys
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"

import torch
import random
import gc
import traceback
from PIL import Image, ImageOps
import gradio as gr

from huggingface_hub import snapshot_download
try:
    from huggingface_hub.errors import LocalEntryNotFoundError
except ImportError:
    from huggingface_hub.utils import LocalEntryNotFoundError

os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ["TRELLIS_CACHE_DIR"] = os.path.join(SYSTEM_DIRS['models'], "trellis_weights")
ATTN_BACKEND_REQUESTED = os.environ.get("AIEROS_TRELLIS_ATTN_BACKEND", "xformers")
if ATTN_BACKEND_REQUESTED not in ("xformers", "flash_attn"):
    raise ValueError(
        f"Unsupported TRELLIS sparse-attention backend {ATTN_BACKEND_REQUESTED!r}; "
        "use xformers or flash_attn (SDPA is not supported by TRELLIS sparse attention)."
    )
os.environ["ATTN_BACKEND"] = ATTN_BACKEND_REQUESTED
os.environ["SPARSE_ATTN_BACKEND"] = ATTN_BACKEND_REQUESTED

# Probe the actual CUDA implementation before importing TRELLIS, so a mismatched
# wheel fails here with an actionable message rather than halfway through generation.
if ATTN_BACKEND_REQUESTED == "xformers":
    try:
        import xformers.ops as xops
        probe = torch.zeros((1, 2, 1, 64), device="cuda", dtype=torch.float16)
        xops.memory_efficient_attention(probe, probe, probe)
        del probe
        print("[ATTENTION] xFormers CUDA operator probe passed.")
    except Exception as exc:
        raise RuntimeError(
            "The installed xFormers CUDA operator is incompatible with this PyTorch/CUDA build. "
            "Run the TRELLIS setup cell in phase2_3d to install the pinned compatible wheel, "
            "or set AIEROS_TRELLIS_ATTN_BACKEND=flash_attn if flash_attn is installed."
        ) from exc
else:
    import flash_attn
    print("[ATTENTION] Using flash_attn for dense and sparse attention.")

from trellis2.pipelines import Trellis2ImageTo3DPipeline
import o_voxel
from storage_manager import managed_storage_operation, storage_preflight
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes
from purge_manager import DiskPurgeManager

OUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "3d_models")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_trellis_pipeline():
    repo_id = TRELLIS_REPO_ID
    hub_cache_dir = os.path.join(SYSTEM_DIRS["models"], "hub")
    try:
        local_model_dir = snapshot_download(
            repo_id=repo_id,
            cache_dir=hub_cache_dir,
            local_files_only=True,
        )
        print(f"TRELLIS cache hit; resolved local snapshot: {local_model_dir}")
    except LocalEntryNotFoundError as cache_error:
        required_gb = (
            estimate_hf_repo_bytes(repo_id) / BYTES_PER_GB
            + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
        )
        min_free_gb = float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0"))
        ok, reason = storage_preflight(
            "models", required_gb=required_gb, require_fresh_quota=True,
            minimum_free_gb=min_free_gb,
        )
        if not ok:
            raise RuntimeError(f"TRELLIS download denied by storage policy: {reason}")
        with managed_storage_operation(
            "models", required_gb=required_gb, require_fresh_quota=True,
            minimum_free_gb=min_free_gb,
        ):
            local_model_dir = snapshot_download(repo_id=repo_id, cache_dir=hub_cache_dir)
    return Trellis2ImageTo3DPipeline.from_pretrained(local_model_dir)
trellis_pipe = load_trellis_pipeline()
trellis_pipe.cuda()

if ENABLE_DISK_PURGE and os.environ.get("AIEROS_ALLOW_SHARED_MODEL_PURGE") == "1":
    print("\n[WARNING] Explicit shared-cache purge requested. This is unsafe if any other process may still be loading the same model.")
    DiskPurgeManager.purge_hf_model("microsoft/TRELLIS.2-4B")
else:
    print("[STORAGE] Shared model/cache files retained; no automatic shared-cache purge.")
# TRELLIS_CACHE_DIR is retained because runtime components may use it lazily.

@torch.inference_mode()
def extrude_3d_asset(input_image, decimation_target: int):
    if input_image is None: 
        raise gr.Error("Please upload an image first!")
        
    seed = random.randint(0, 999999)
    print(f"\n[EXTRUDING] Processing uploaded image | Seed: {seed}")
    
    try:
        input_image = input_image.convert("RGB")
        w, h = input_image.size
        max_size = 1024
        
        if max(w, h) > max_size:
            scale = max_size / float(max(w, h))
            w = int(w * scale)
            h = int(h * scale)
            input_image = input_image.resize((w, h), Image.Resampling.LANCZOS)
            
        sq_size = max(w, h)
        sq_size = max(64, int(round(sq_size / 64.0)) * 64)
        
        square_img = Image.new("RGB", (sq_size, sq_size), (255, 255, 255))
        square_img.paste(input_image, ((sq_size - w) // 2, (sq_size - h) // 2))
        input_image = square_img
            
        torch.cuda.empty_cache()
            
        mesh = trellis_pipe.run(input_image)[0]
        mesh.simplify(16777216) 
        
        glb_path = os.path.join(OUTPUT_DIR, f"standalone_t2_{seed}.glb")
        glb = o_voxel.postprocess.to_glb(
            vertices            =   mesh.vertices,
            faces               =   mesh.faces,
            attr_volume         =   mesh.attrs,
            coords              =   mesh.coords,
            attr_layout         =   mesh.layout,
            voxel_size          =   mesh.voxel_size,
            aabb                =   [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
            decimation_target   =   int(decimation_target),
            texture_size        =   4096,
            remesh              =   True,
            remesh_band         =   1,
            remesh_project      =   0,
            verbose             =   False
        )
        with managed_storage_operation(
            "outputs",
            required_gb=0.20,
            require_fresh_quota=False,
            minimum_free_gb=0.05,
        ):
            glb.export(glb_path, extension_webp=True)
        
        del mesh, glb
        gc.collect()
        torch.cuda.empty_cache()
        return glb_path
        
    except Exception as e:
        error_trace = traceback.format_exc()
        print(f"\n[CRITICAL ERROR]\n{error_trace}")
        gc.collect()
        torch.cuda.empty_cache()
        raise gr.Error(f"Generation Failed: {str(e)}")

with gr.Blocks(title="AIEROS TRELLIS.2 Standalone") as demo:
    gr.Markdown("# 🕹️ AIEROS Studio: TRELLIS.2-4B (Isolated Testing)")
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label="Upload 2D Concept", type="pil")
            decimation = gr.Slider(label="Target Polygon Count", minimum=10000, maximum=500000, value=100000, step=10000)
            generate_btn = gr.Button("Extrude to 3D", variant="primary", size="lg")
        with gr.Column(scale=2):
            output_3d = gr.Model3D(label="TRELLIS.2 3D Viewer", clear_color=[0.1, 0.1, 0.1, 1.0])

    generate_btn.click(fn=extrude_3d_asset, inputs=[input_image, decimation], outputs=[output_3d])

print("Starting Gradio Web Server on port 7862...")
demo.launch(server_name="0.0.0.0", server_port=7862, inline=False, share=False, show_error=True)


In [12]:
# demo.close()

In [13]:
!nvidia-smi